In [19]:
import joblib

In [20]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../dataset/processed/feature_engineered_house_prices.csv"
)

print(df.shape)

(2997, 12)


In [21]:
features = [
    "BEDS",
    "BATH",
    "PROPERTYSQFT",
    "TYPE",
    "STATE",
    "LOCALITY"
]

X = df[features]
y = df["PRICE"]

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [24]:
categorical_features = [
    "TYPE",
    "STATE",
    "LOCALITY"
]

numerical_features = [
    "BEDS",
    "BATH",
    "PROPERTYSQFT"
]

In [25]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=5
            )
        )
    ]
)

numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numerical",
            numerical_transformer,
            numerical_features
        )
    ]
)

In [26]:
import json

with open(
    "../dataset/processed/best_xgb_params.json",
    "r"
) as f:
    best_params = json.load(f)

print(best_params)

{'model__subsample': 0.9, 'model__n_estimators': 200, 'model__min_child_weight': 5, 'model__max_depth': 8, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.7}


In [27]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=best_params["model__n_estimators"],
    learning_rate=best_params["model__learning_rate"],
    max_depth=best_params["model__max_depth"],
    min_child_weight=best_params["model__min_child_weight"],
    subsample=best_params["model__subsample"],
    colsample_bytree=best_params["model__colsample_bytree"],
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

In [28]:
final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

In [29]:
final_model.fit(
    X_train,
    y_train
)

print("Final model trained successfully.")

Final model trained successfully.


In [30]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [31]:
y_pred = final_model.predict(X_test)

In [32]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

In [33]:
print("FINAL MODEL RESULTS")
print("-------------------")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

FINAL MODEL RESULTS
-------------------
MAE  : 16,928,736.36
RMSE : 27,864,528.77
R²   : 0.8691


In [35]:
joblib.dump(
    final_model,
    "../models/house_price_xgboost.pkl"
)

print("Model saved successfully.")

Model saved successfully.


In [36]:
loaded_model = joblib.load(
    "../models/house_price_xgboost.pkl"
)

print("Model loaded successfully.")

Model loaded successfully.
